# YOLOv8 vs YOLO11 vs YOLO26 vs Roboflow Overhead Person

This notebook compares the local `yolov8n.pt`, `yolo11n.pt`, and `YOLO("yolo26n.pt")` on one image.

If you set a `ROBOFLOW_API_KEY` environment variable or paste a key into the notebook, it also runs the Roboflow hosted model `overhead-person-szky0/3`.

Drop your target image at:

`datasets/raw/web_bottom_challenge_v1/images/target_compare.jpg`

If that file is missing, the notebook falls back to a demo image so the cells still run.

In [ ]:
from __future__ import annotations

import base64
import json
import os
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from PIL import Image, ImageDraw
from ultralytics import YOLO


In [ ]:
# Optional: paste your Roboflow API key here if it is not available as an environment variable.
ROBOFLOW_API_KEY_OVERRIDE = ''

# Optional: if you already downloaded yolo26n.pt, paste the full path here.
# If left empty, the notebook will try YOLO('yolo26n.pt').
YOLO26_WEIGHT_OVERRIDE = ''


In [ ]:
ROOT = Path.cwd()
if not (ROOT / 'MovingDroneCrowd').exists():
    ROOT = ROOT.parent

TARGET_IMAGE_PATH = ROOT / 'datasets/raw/web_bottom_challenge_v1/images/target_compare.jpg'
FALLBACK_IMAGE_PATH = ROOT / 'LOAF/resolution_1k/test/0053_00060.jpg'

if TARGET_IMAGE_PATH.exists():
    IMAGE_PATH = TARGET_IMAGE_PATH
    IMAGE_NOTE = 'Using target_compare.jpg from datasets/raw/web_bottom_challenge_v1/images/'
else:
    IMAGE_PATH = FALLBACK_IMAGE_PATH
    IMAGE_NOTE = 'Target image not found. Using the LOAF demo image instead. Drop your image at datasets/raw/web_bottom_challenge_v1/images/target_compare.jpg and rerun.'

YOLO26_CANDIDATES = [
    Path(YOLO26_WEIGHT_OVERRIDE) if YOLO26_WEIGHT_OVERRIDE else None,
    ROOT / 'MovingDroneCrowd/yolo26n.pt',
    ROOT / 'weights/yolo26n.pt',
]

def _first_existing(paths):
    for path in paths:
        if path and path.exists():
            return path
    return None

YOLO26_LOCAL_PATH = _first_existing(YOLO26_CANDIDATES)
YOLO26_SOURCE = str(YOLO26_LOCAL_PATH) if YOLO26_LOCAL_PATH else (YOLO26_WEIGHT_OVERRIDE.strip() or 'yolo26n.pt')

MODEL_SOURCES = {
    'YOLOv8n': str(ROOT / 'MovingDroneCrowd/yolov8n.pt'),
    'YOLO11n': str(ROOT / 'MovingDroneCrowd/yolo11n.pt'),
    'YOLO26n': YOLO26_SOURCE,
}

ROBOFLOW_MODEL_ID = 'overhead-person-szky0/3'
ROBOFLOW_API_URL = f'https://detect.roboflow.com/{ROBOFLOW_MODEL_ID}'
ROBOFLOW_API_KEY = (ROBOFLOW_API_KEY_OVERRIDE or os.getenv('ROBOFLOW_API_KEY', '')).strip()

CONF = 0.40
IOU = 0.70
IMGSZ = 960
DEVICE = 'cpu'

def describe_source(source: str) -> str:
    path = Path(source)
    if source == 'yolo26n.pt':
        return 'model-name download'
    if path.exists():
        return 'local file'
    return 'missing local file'

display(Markdown(f'**Workspace Root:** `{ROOT}`'))
display(Markdown(f'**Image:** `{IMAGE_PATH}`'))
display(Markdown(f'**Note:** {IMAGE_NOTE}'))
display(pd.DataFrame({
    'model': list(MODEL_SOURCES.keys()) + ['Roboflow Overhead Person'],
    'source': list(MODEL_SOURCES.values()) + [ROBOFLOW_API_URL],
    'mode': [describe_source(source) for source in MODEL_SOURCES.values()] + [('api ready' if ROBOFLOW_API_KEY else 'api key missing')],
})))

if not ROBOFLOW_API_KEY:
    display(Markdown('**Roboflow note:** The hosted Roboflow model is disabled because no API key was provided. Paste a key into `ROBOFLOW_API_KEY_OVERRIDE` or set `ROBOFLOW_API_KEY` before rerunning.'))

if YOLO26_SOURCE == 'yolo26n.pt':
    display(Markdown('**YOLO26 note:** The notebook will try `YOLO("yolo26n.pt")`. If download is blocked, put the file at `C:/Users/PC/Desktop/Term252/Senior Project II/MovingDroneCrowd/yolo26n.pt` or set `YOLO26_WEIGHT_OVERRIDE`.'))


In [ ]:
def render_prediction(image_path: Path, predictions: list[dict[str, float]], title: str):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)

    for pred in predictions:
        x1 = int(pred['x1'])
        y1 = int(pred['y1'])
        x2 = int(pred['x2'])
        y2 = int(pred['y2'])
        conf = float(pred.get('confidence', 0.0))
        label = str(pred.get('label', 'person'))
        draw.rectangle((x1, y1, x2, y2), outline=(0, 220, 0), width=3)
        text_y = y1 - 16 if y1 >= 18 else y1 + 4
        draw.text((x1 + 2, text_y), f'{label} {conf:.2f}', fill=(255, 255, 0))

    draw.rectangle((8, 8, 560, 68), outline=(255, 255, 255), width=2)
    draw.text((16, 18), title, fill=(255, 255, 255))
    draw.text((16, 42), f'Detections: {len(predictions)}', fill=(255, 255, 255))
    return image


def ultralytics_predictions(result) -> list[dict[str, float]]:
    if result.boxes is None or len(result.boxes) == 0:
        return []

    boxes = result.boxes.xyxy.cpu().numpy()
    confs = result.boxes.conf.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy() if result.boxes.cls is not None else [0] * len(boxes)
    names = result.names if hasattr(result, 'names') else {}

    predictions = []
    for xyxy, conf, cls_id in zip(boxes, confs, classes):
        x1, y1, x2, y2 = [float(v) for v in xyxy]
        predictions.append({
            'x1': x1,
            'y1': y1,
            'x2': x2,
            'y2': y2,
            'confidence': float(conf),
            'label': str(names.get(int(cls_id), int(cls_id))),
        })
    return predictions


def roboflow_predictions(payload: dict) -> list[dict[str, float]]:
    predictions = []
    for pred in payload.get('predictions', []):
        x = float(pred['x'])
        y = float(pred['y'])
        width = float(pred['width'])
        height = float(pred['height'])
        predictions.append({
            'x1': x - (width / 2.0),
            'y1': y - (height / 2.0),
            'x2': x + (width / 2.0),
            'y2': y + (height / 2.0),
            'confidence': float(pred.get('confidence', 0.0)),
            'label': str(pred.get('class', 'person')),
        })
    return predictions


def predict_with_ultralytics(model_source: str, image_path: Path, conf: float):
    path = Path(model_source)
    if model_source != 'yolo26n.pt' and model_source.endswith('.pt') and ('/' in model_source or '\\' in model_source) and not path.exists():
        raise FileNotFoundError(model_source)
    model = YOLO(model_source)
    result = model.predict(str(image_path), conf=conf, iou=IOU, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
    return ultralytics_predictions(result)


def infer_local_model(model_name: str, model_source: str, image_path: Path):
    try:
        predictions = predict_with_ultralytics(model_source, image_path, CONF)
    except FileNotFoundError:
        return {
            'model': model_name,
            'source': model_source,
            'detections': None,
            'mean_confidence': None,
            'status': 'skipped: missing local weights',
            'rendered': None,
        }
    except Exception as exc:
        return {
            'model': model_name,
            'source': model_source,
            'detections': None,
            'mean_confidence': None,
            'status': f'error: {exc}',
            'rendered': None,
        }

    mean_conf = 0.0 if not predictions else sum(item['confidence'] for item in predictions) / len(predictions)
    return {
        'model': model_name,
        'source': model_source,
        'detections': len(predictions),
        'mean_confidence': mean_conf,
        'status': 'ok',
        'rendered': render_prediction(image_path, predictions, model_name),
    }


def infer_roboflow_model(image_path: Path):
    if not ROBOFLOW_API_KEY:
        return {
            'model': 'Roboflow Overhead Person',
            'source': ROBOFLOW_API_URL,
            'detections': None,
            'mean_confidence': None,
            'status': 'skipped: missing ROBOFLOW_API_KEY',
            'rendered': None,
        }

    encoded_image = base64.b64encode(image_path.read_bytes())
    params = urllib.parse.urlencode({
        'api_key': ROBOFLOW_API_KEY,
        'confidence': int(CONF * 100),
        'overlap': int(IOU * 100),
    })
    request = urllib.request.Request(
        f'{ROBOFLOW_API_URL}?{params}',
        data=encoded_image,
        headers={'Content-Type': 'application/x-www-form-urlencoded'},
        method='POST',
    )

    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            payload = json.loads(response.read().decode('utf-8'))
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode('utf-8', errors='replace')
        return {
            'model': 'Roboflow Overhead Person',
            'source': ROBOFLOW_API_URL,
            'detections': None,
            'mean_confidence': None,
            'status': f'http_error {exc.code}: {detail[:180]}',
            'rendered': None,
        }
    except Exception as exc:
        return {
            'model': 'Roboflow Overhead Person',
            'source': ROBOFLOW_API_URL,
            'detections': None,
            'mean_confidence': None,
            'status': f'error: {exc}',
            'rendered': None,
        }

    predictions = roboflow_predictions(payload)
    mean_conf = 0.0 if not predictions else sum(item['confidence'] for item in predictions) / len(predictions)
    return {
        'model': 'Roboflow Overhead Person',
        'source': ROBOFLOW_API_URL,
        'detections': len(predictions),
        'mean_confidence': mean_conf,
        'status': 'ok',
        'rendered': render_prediction(image_path, predictions, 'Roboflow Overhead Person'),
    }


In [ ]:
records = [infer_local_model(name, source, IMAGE_PATH) for name, source in MODEL_SOURCES.items()]
rf_record = infer_roboflow_model(IMAGE_PATH)
records.append(rf_record)

comparison_df = pd.DataFrame([
    {
        'model': item['model'],
        'detections': item['detections'],
        'mean_confidence': item['mean_confidence'],
        'status': item['status'],
        'source': item['source'],
    }
    for item in records
])
display(comparison_df)


In [ ]:
visual_records = [item for item in records if item['rendered'] is not None]
fig, axes = plt.subplots(1, len(visual_records), figsize=(10 * len(visual_records), 10))
if len(visual_records) == 1:
    axes = [axes]
for ax, item in zip(axes, visual_records):
    ax.imshow(item['rendered'])
    mean_conf = 0.0 if item['mean_confidence'] is None else item['mean_confidence']
    ax.set_title(f"{item['model']} | detections={item['detections']} | mean_conf={mean_conf:.3f}")
    ax.axis('off')
plt.tight_layout()


In [ ]:
sweep_records = []
for conf in [0.05, 0.10, 0.15, 0.20, 0.25, 0.40]:
    row = {'conf': conf}
    for model_name, model_source in MODEL_SOURCES.items():
        try:
            predictions = predict_with_ultralytics(model_source, IMAGE_PATH, conf)
            row[model_name] = len(predictions)
        except Exception:
            row[model_name] = None
    sweep_records.append(row)
display(pd.DataFrame(sweep_records))

display(Markdown('**Note:** The confidence sweep only covers the local Ultralytics models. Roboflow is intentionally excluded to avoid repeated hosted API calls.'))
